## Logistic regression: Overfitting
- make_moons is a a scikit-learn function that generates a synthetic, 2-dimensional dataset shaped like two interleaving half-circles

In [ ]:
# For this notebooks:
# python -m venv .venv
# source .venv/bin/activate
# pip install numpy matplotlib scikit-learn typing ipykernel 

### Fit with libraries to appreciate results

Below we will fit the example that was shown in the lecture using very commonly used `sklearn` methods. We define a `Pipeline` that helps us run multiple steps, such as making polynomoial features, scale them and fit the logistic regression. We will see (full disclosure: I had to slightly fine tune the example's parameters to show this) that the overfitted model has a larger training accuracy but a smaller test accuracy when compared to the regularized one. Also, as will be shown in the tutorial, it takes many more iterations to converge, which is what people in numerical computing call an "ill-conditioned" problem.

In [ ]:
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none' # Keep text as actual text elements in the SVG
%config InlineBackend.figure_format = 'svg'
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# 1. Generate synthetic data
X, y = make_moons(n_samples=750, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Define the three models
TOL = 1e-6
POLY_DEG = 15

# Model 1: Underfitting / Too Simple (Linear Decision Boundary)
# Note: In scikit-learn, C is inverse regularization strength. C=1e5 means practically no penalty.
model_underfit = LogisticRegression(C=1e5, tol=TOL)

# Model 2: Overfitting (High degree polynomial, no regularization)
# Note: we scale features by subtracting the mean and dividing by the standard deviation, to standardize the data after the polynomial transformation
model_overfit = Pipeline([
    ("poly_features", PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
    ("scale", StandardScaler()),
    ("log_reg", LogisticRegression(C=1e5, max_iter=30000, tol=TOL))
])

# Model 3: Regularized / Just Right (Same high degree, but strong L2 penalty)
# Lower C value = stronger regularization
model_regularized = Pipeline([
    ("poly_features", PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
    ("scale", StandardScaler()),
    ("log_reg", LogisticRegression(C=0.5, max_iter=10000, tol=TOL))
])

models = [model_underfit, model_overfit, model_regularized]
titles = ["Underfitting\n(Purely Linear)", f"Overfitting\n(Degree {POLY_DEG}, No Penalty)", f"Regularized\n(Degree {POLY_DEG}, L2 Penalty)"]

# 3. Plotting setup
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

# Create a mesh grid to plot decision boundaries
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

for ax, model, title in zip(axes, models, titles):
    # Fit the model
    model.fit(X_train, y_train)

    # n_iter_ is the number of optimizer iterations used before convergence.
    if isinstance(model, Pipeline):
        iterations = model.named_steps["log_reg"].n_iter_.max()
    else:
        iterations = model.n_iter_.max()

    # Evaluate performance
    train_acc = model.score(X_train, y_train) * 100
    test_acc = model.score(X_test, y_test) * 100
    
    # Plot decision boundary
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap=plt.cm.Paired)
    
    # Plot training points (solid) and test points (semi-transparent)
    ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='red', label='Train Class 0', edgecolors='k')
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='blue', label='Train Class 1', edgecolors='k')
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], c='red', alpha=0.4, marker='x', label='Test Class 0')
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], c='blue', alpha=0.4, marker='x', label='Test Class 1')
    
    # Formatting
    subfig_text =  f"Train Acc: {train_acc:.1f}%\n" \
        f"Test Acc: {test_acc:.1f}%\n" \
        f"Iterations: {iterations}"
    #print(f"\nModel {title} yields:\n",subfig_text)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.text(
        x_min + 0.1,
        y_min + 0.4,
        subfig_text,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
        fontsize=11,
    )
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

axes[0].legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("lr-overfit.svg", format="svg", bbox_inches="tight")
plt.close()

## Exercise: Gradient of Regularization
The whole exercise part of the notebook is rather short, simply write the regularization function and its gradient for the 3 different techniques: L2 (ridge), L1 (lasso), and elastic net (an average of the previous two). We won't really use the regularized cost function explicitly, but it's still useful to write it. You can try to use it as a stopping criterion instead of the gradient norm and see what changes. Read the comments at the end and play a bit around with the code. More about regularization in the turorial after the next lecture. 

In [ ]:
import numpy as np
import numpy.typing as npt

def cost_function_regularized(
    X: npt.NDArray, 
    y: npt.NDArray, 
    w: npt.NDArray, 
    b: float,
    lambda_: float = 0.1,
    penalty: str = "l2"
) -> float:
    m = len(y)
    cost_sum = 0.0

    # Your original loss loop
    for i in range(m):
        z = float(np.dot(w, X[i]) + b)
        g = sigmoid(z)
        # Clamping g slightly avoids log(0) errors
        g = np.clip(g, 1e-15, 1 - 1e-15) 
        cost_sum += -y[i] * np.log(g) - (1 - y[i]) * np.log(1 - g)
    
    total_cost = (1 / m) * cost_sum

    # Add the regularization penalty
    if penalty == "l2":
        # Ridge: (lambda / 2m) * sum(w^2)
        # TODO
        reg_term = 0.0
        total_cost += reg_term
    elif penalty == "l1":
        # Lasso: (lambda / m) * sum(|w|)
        # TODO
        reg_term = 0.0
        total_cost += reg_term
    elif penalty == "elasticnet":
        # Combination (50/50 split example)
        # TODO
        l1 = 0.0
        # TODO
        l2 = 0.0
        total_cost += 0.5 * l1 + 0.5 * l2

    return total_cost

In [ ]:
def compute_gradient_regularized(
    X: npt.NDArray, 
    y: npt.NDArray, 
    w: npt.NDArray, 
    b: float,
    lambda_: float = 0.1,
    penalty: str = "l2"
) -> tuple[npt.NDArray, float]:
    # 1. Compute standard unregularized gradient. The first part is identical to gradient_function(), 
    # but uses vectoried numpy for efficiency (too slow otherwise)
    m = len(y)

    probabilities = sigmoid(X @ w + b)
    errors = probabilities - y

    # These are exactly the sums from the nested loops, written as matrix ops.
    grad_w = (X.T @ errors) / m
    grad_b = np.mean(errors)

    # 2. Add regularization derivative (Skip bias 'db', as is usually done), with coefficient lambda_
    # In this case, lambda_ is the regularization strength, i.e. the inverse of the previous `C``
    # Note: the underscore is needed since lambda is a python keyword, like `if`. 
    if penalty == "l2":
        # TODO
    elif penalty == "l1":
        # TODO
    elif penalty == "elasticnet":
        # TODO


    return grad_w, grad_b

## Handwritten optimization with polynomial features

*Markdown comment AI-generated by GPT-5.6 Terra via Copilot SDK in VS Code.*

We use `PolynomialFeatures` and `StandardScaler` only to prepare the data. The learning algorithm is handwritten: it computes the regularized logistic-regression gradient, updates the parameters with gradient descent, and stops when the gradient norm is sufficiently small.

For L2 regularization in this notebook, the handwritten parameter is the inverse of scikit-learn's `C` parameter:

\[\lambda = \frac{1}{C}\]

Therefore, `lambda_=0.01` corresponds to `C=100`, while `lambda_=2.0` corresponds to `C=0.5`.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def fit_by_gradient_descent(
    X, y, lambda_, penalty, learning_rate=0.5, max_iter=540_000, tol=1e-6
):
    w = np.zeros(X.shape[1])
    b = 0.0

    for iteration in range(max_iter):
        grad_w, grad_b = compute_gradient_regularized(
            X, y, w, b, lambda_=lambda_, penalty=penalty
        )

        w -= learning_rate * grad_w
        b -= learning_rate * grad_b

       
        gradient_norm = np.sqrt(np.sum(grad_w ** 2) + grad_b ** 2)
        # Stopping when gradient smaller than tolerance as sole criterion, could be dangerous...
        if gradient_norm < tol:
            break

    final_grad_w, final_grad_b = compute_gradient_regularized(
    X, y, w, b, lambda_=lambda_, penalty=penalty
    )
    final_gradient_norm = np.hypot(np.linalg.norm(final_grad_w), final_grad_b)

    return w, b, iteration + 1, final_gradient_norm

## Training Loop
Now we are done defining stuff, this is where the training is carried out!

In [ ]:
# Use the same degree-15 representation as the earlier scikit-learn example.

# NOTE: if this is too slow on your machine, (1) focus on weak penalty vs L2, (2) make only polynomial features up to 5 or 10, not up to 15
#  (3) use fewer maximum number of training iteration, (4) make the tolerance larger 

from time import perf_counter

poly = PolynomialFeatures(degree=POLY_DEG, include_bias=False)
scaler = StandardScaler()

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

X_train_poly = scaler.fit_transform(X_train_poly)
X_test_poly = scaler.transform(X_test_poly)



models_to_compare = [
    ("Weak penalty", "l2", 1e-2),
    ("L2 penalty", "l2", 2.0),
    ("L1 penalty", "l1", 2.0),
    ("Elastic net", "elasticnet", 2.0),
]

print(f"{'Model':<15} {'Train acc':>10} {'Test acc':>10} {'||w||':>10} {'Steps':>8} {'||grad||':>14} {'Time (s)':>10}")
print("-" * 85)

for name, penalty, lambda_ in models_to_compare:
    start_time = perf_counter()
    w, b, steps, final_gradient_norm = fit_by_gradient_descent(
        X_train_poly,
        y_train,
        lambda_=lambda_,
        penalty=penalty,
    )
    training_time = perf_counter() - start_time

    train_predictions = (sigmoid(X_train_poly @ w + b) >= 0.5).astype(int)
    test_predictions = (sigmoid(X_test_poly @ w + b) >= 0.5).astype(int)

    train_accuracy = np.mean(train_predictions == y_train) * 100
    test_accuracy = np.mean(test_predictions == y_test) * 100
    weight_norm = np.linalg.norm(w)

    print(
        f"{name:<15} {train_accuracy:>9.1f}% {test_accuracy:>9.1f}% "
        f"{weight_norm:>10.3f} {steps:>8} {final_gradient_norm:14.2e} {training_time:>10.2f}"
    )

Model            Train acc   Test acc      ||w||    Steps       ||grad||   Time (s)
-------------------------------------------------------------------------------------
Weak penalty         93.7%      94.2%     27.097   515602       1.00e-06      12.69
L2 penalty           91.4%      94.7%      4.097     5473       9.98e-07       0.14
L1 penalty           92.6%      94.7%      9.171   540000       4.08e-02      13.83
Elastic net          91.4%      94.7%      4.461   540000       1.91e-02      14.61


## Interpreting the handwritten results

*Markdown comment AI-generated by GPT-5.6 Terra via Copilot SDK in VS Code.*

The weakly regularized model has higher training accuracy, but slightly lower test accuracy than the more strongly L2-regularized model. This illustrates the bias-variance trade-off: allowing larger coefficients can fit the training data more closely, but may also fit details that do not generalize to unseen data.

The L2 penalty directly discourages large coefficients. This is visible in the smaller value of ||w|| for the L2-regularized model.

The regularized model also reaches a small gradient norm in far fewer gradient-descent steps. Regularization can therefore help both generalization and the numerical behavior of the optimization problem.

The test-accuracy difference in this synthetic example is small, so it is an illustration of the trade-off rather than a universal rule: regularization does not always improve test accuracy.

### Note on L1 and elastic-net optimization

*Markdown comment AI-generated by GPT-5.6 Terra via Copilot SDK in VS Code.*

The L1 penalty is not differentiable at zero. The simple `np.sign(w)` subgradient used here demonstrates the mathematical idea, but it is not the optimizer normally used in practice. Efficient L1 and elastic-net methods typically use coordinate descent, proximal gradient descent, or SAGA.

The L1 and elastic-net rows are therefore optional extensions. The weak-L2 versus L2 comparison is the main result of this notebook.